# RandomForestRegressor → fail-closed decision model

**Problem.** Predict disease progression (`diabetes`, all 10 features) and decide whether the prediction is
*safe enough to act on* at a declared threshold of **240**. `sklearn`'s diabetes features are
mean-centred and scaled, so all thresholds below are in standardized units.

**Why a harness, not just a model.** A regressor always returns a number. A decision model must answer
*"may this run?"* — and must refuse when the number is not trustworthy. `mjolnir_ml` wraps the fitted
RandomForest with:

- a **regime floor** — inputs outside the training support abstain;
- a **conformal κ** — an interval with finite-sample coverage at α = 0.10, tightened
  from a constant split band to a locally adaptive CQR band;
- a **declared policy** (`@dsl`) over the interval: release only if the whole interval clears the band,
  deny only if the whole interval is above it, ask otherwise;
- a **certificate** — claim/policy/model/calibration/risk digests, replayable without the model;
- a **risk track** — Conformal Risk Control over a crossing model `P(Y>240|x)` for action-level
  release/deny guarantees, with a feasibility audit that names dead rules.

**How to read the DSL.** Statements see two kinds of facts: *stated input facts* (features such as `bmi`)
and *model facts* supplied by the harness (`pred`, `lo`, `hi`). A **premise** that fails makes
the row undecidable (`abstain`); a **rule** that fires escalates (`ask`) or refuses (`deny`); a **release**
statement is the only way to release.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # the mjolnir_ml package

import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from mjolnir_ml import (ASK, DENY, RELEASE, DecisionModel, DecisionPolicyBase,
                        HazardLedger, dsl, report, rule, certify, verify_certificate)

pd.set_option("display.width", 140)

## 1. The learner and the data (all features)

All 10 features are kept: the forest sees the full representation and the policy may read any of them as
stated facts. `Xcal` is a **held-out calibration split** — conformal calibration must not see training
rows.

In [2]:
data = load_diabetes()
FEATURES = tuple(data.feature_names)
print("features:", FEATURES)
print("n =", len(data.data), " target range:", data.target.min(), "-", data.target.max())

Xtr, Xrest, ytr, yrest = train_test_split(data.data, data.target, test_size=0.4, random_state=0)
Xcal, Xte, ycal, yte = train_test_split(Xrest, yrest, test_size=0.5, random_state=0)

rows = lambda A: [dict(zip(FEATURES, map(float, x))) for x in A]
train_rows, cal_rows, test_rows = rows(Xtr), rows(Xcal), rows(Xte)

forest = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
pd.Series(forest.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

features: ('age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6')
n = 442  target range: 25.0 - 346.0


s5     0.363
bmi    0.239
bp     0.105
s6     0.057
age    0.055
s3     0.054
s1     0.045
s2     0.045
s4     0.022
sex    0.015
dtype: float64

## 2. The policy (compiled from source)

- `bmi_in_range` — a **premise** (sanity bound on the standardized BMI): a void input cannot be decided.
- `release_safe_band` — the *entire* conformal interval is below 240 → `release`.
  Only the **upper** bound matters: progression cannot be negative, so a statistical
  lower bound below 0 is not a safety signal (requiring `lo >= 0` left the safest,
  lowest predictions uncovered).
- `deny_high_band` — the *entire* interval is above 240 → `deny` (nobody may grant this).
- `review_widening` — the interval straddles 240 → `ask` (a human decides).

These three rules partition the line: `hi <= 240`, `lo > 240`, or neither.
There is no gap left for an `abstain(uncovered)`.

The policy never sees the forest object; it sees `lo`/`hi` (conformal) and stated facts.

In [3]:
@dsl
class ProgressionPolicy(DecisionPolicyBase):
    facts = ("age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6")

    assert lambda bmi: bmi > -1.0, "bmi_in_range"

    assert rule("release_safe_band",
                when=lambda hi: hi <= 240.0, verdict=RELEASE)
    assert rule("deny_high_band", when=lambda lo: lo > 240.0, verdict=DENY)
    assert rule("review_widening",
                when=lambda lo, hi: lo <= 240.0 and hi > 240.0, verdict=ASK)

ProgressionPolicy.policy()

Policy('ProgressionPolicy', 4 statements, sha256:ebd516924fed475...)

`fit_regime` records the training support; `calibrate` fits the standard split-conformal index at
α = 0.10. One row is then disposed: verdict, basis, guarantee tag and certificate id are the product.
Section 4 then tightens the same interval without changing the guarantee.

In [4]:
dm = (DecisionModel(forest, ProgressionPolicy, features=FEATURES)
      .fit_regime(train_rows, method="mahalanobis",
                        alpha_ood=0.02, calibration_rows=cal_rows)
      .calibrate(cal_rows, ycal, alpha=0.10))

d = dm.dispose(test_rows[0])
print("verdict :", d.verdict)
print("basis   :", list(d.basis))
print("say     :", d.say)
print("facts   :", {k: round(v, 3) for k, v in d.prediction.items() if isinstance(v, float)})
print("cert id :", d.certificate_id[:34], "…")
print("replay  :", dm.replay(d, test_rows[0]))

verdict : ask
basis   : ['review_widening']
say     : escalated by review_widening (lo=143.62666666666667, hi=360.94666666666666)
facts   : {'pred': 252.287, 'lo': 143.627, 'hi': 360.947}
cert id : sha256:52504b421c38c69f17a529ef350 …
replay  : True


### Gates in action

The pipeline order is **contract → regime → predict → premises → rules**, and each gate fails closed.
Feed an implausible BMI and the *regime floor* refuses before the forest is called: the harness never
extrapolates outside the recorded training support. (Premises sit after the regime check and catch
states that are individually in-range but jointly invalid — e.g. `assert lambda bmi, bp: not (bmi > 0.15 and bp < 0.0)`.)

In [5]:
bad = dict(test_rows[0], bmi=-5.0)
d_bad = dm.dispose(bad)
print(d_bad.verdict, "|", list(d_bad.basis), "|", d_bad.say)

abstain | ['out_of_regime:mahalanobis'] | cannot decide: outside the multivariate regime (mahalanobis score=20596.4 > tau=1198.82)


## 4. Tightening the interval (without weakening the guarantee)

The split band `pred ± q` is constant: it pays the worst-case residual everywhere. Two local-width
variants keep the same finite-sample coverage and only move the width:

| method | nonconformity score | interval |
|---|---|---|
| `split` | `|y − f(x)|` | `[f(x) − q, f(x) + q]` |
| `normalized` | `|y − f(x)| / σ(x)` | `[f(x) − q·σ(x), f(x) + q·σ(x)]` |
| `cqr` | `max(l(x) − y, y − u(x))` | `[l(x) − q, u(x) + q]` |

`normalized` uses a residual model fitted on **out-of-fold** training errors (never the calibration
split). `cqr` uses fitted lower/upper quantile models (gradient boosting with quantile loss). Both go
through the same `fit_kappa` floor and digest discipline; only the width changes.


In [6]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

def width_census(model):
    ds = [model.dispose(r) for r in test_rows]
    pairs = [(d, y) for d, y in zip(ds, yte)
             if d.prediction and "lo" in d.prediction]
    widths = [d.prediction["hi"] - d.prediction["lo"] for d, _ in pairs]
    cover = np.mean([d.prediction["lo"] <= y <= d.prediction["hi"]
                     for d, y in pairs])
    return (float(np.mean(widths)), float(cover),
            sum(d.verdict == RELEASE for d in ds),
            sum(d.verdict == ASK for d in ds))

# normalized: scale = fitted model of |residual|, trained out-of-fold
oof = cross_val_predict(
    RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    Xtr, ytr, cv=5)
scale = RandomForestRegressor(n_estimators=300, min_samples_leaf=10,
                              random_state=0, n_jobs=-1).fit(Xtr, abs(ytr - oof))
dm_norm = (DecisionModel(forest, ProgressionPolicy, features=FEATURES)
           .fit_regime(train_rows, method="mahalanobis",
                        alpha_ood=0.02, calibration_rows=cal_rows)
           .calibrate(cal_rows, ycal, alpha=0.10,
                      method="normalized", scale=scale))

# CQR: lower/upper quantile models (5% / 95%) + conformal correction q
lower = GradientBoostingRegressor(loss="quantile", alpha=0.05,
                                  random_state=0).fit(Xtr, ytr)
upper = GradientBoostingRegressor(loss="quantile", alpha=0.95,
                                  random_state=0).fit(Xtr, ytr)
dm_cqr = (DecisionModel(forest, ProgressionPolicy, features=FEATURES)
          .fit_regime(train_rows, method="mahalanobis",
                        alpha_ood=0.02, calibration_rows=cal_rows)
          .calibrate(cal_rows, ycal, alpha=0.10,
                     method="cqr", lower=lower, upper=upper))

print(f"{'method':<12}{'mean width':>12}{'coverage':>10}{'release':>9}{'ask':>6}")
for name, model in [("split", dm), ("normalized", dm_norm), ("cqr", dm_cqr)]:
    w, cover, rel, ask = width_census(model)
    print(f"{name:<12}{w:>12.1f}{cover:>10.3f}{rel:>9}{ask:>6}")

dm = dm_cqr      # the tightened interval governs the rest of the notebook
print("\nchosen:", dm.kappa.method, "· calibration", dm.kappa.digest[:22] + "…")


method        mean width  coverage  release   ask


split              217.3     0.943       30    57


normalized         212.9     0.931       30    57


cqr                197.3     0.943       33    54

chosen: cqr · calibration sha256:5156dca8263a3b7…


### Two-track bounds: independent release and deny tails

The release rule reads `hi` and the deny rule reads `lo`, so the two tails need not share one margin.
`one_sided=True` calibrates each tail at its own level: `P(Y >= lo) >= 1 - alpha_lower` and
`P(Y <= hi) >= 1 - alpha_upper`. This is a **decision envelope**, not a joint interval — by the
union bound the joint coverage is only `1 - alpha_lower - alpha_upper`.

The important consequence is not "deny suddenly fires": a deny needs a 90% claim that progression is
above 240, i.e. the lower conditional decile must clear 240. The table below reports what the data
actually supports.


In [7]:
dm_1s = (DecisionModel(forest, ProgressionPolicy, features=FEATURES)
           .fit_regime(train_rows, method="mahalanobis",
                        alpha_ood=0.02, calibration_rows=cal_rows)
           .calibrate(cal_rows, ycal, alpha=0.10, method="cqr", lower=lower, upper=upper,
                      one_sided=True, alpha_lower=0.10, alpha_upper=0.10))

ds = [dm_1s.dispose(r) for r in test_rows]
pairs = [(d, y) for d, y in zip(ds, yte) if d.prediction and "lo" in d.prediction]
los = np.array([d.prediction["lo"] for d, _ in pairs])
his = np.array([d.prediction["hi"] for d, _ in pairs])
ys = np.array([y for _, y in pairs])
print(f"band {np.mean(his - los):.1f} · release "
      f"{sum(d.verdict == RELEASE for d in ds)} · ask "
      f"{sum(d.verdict == ASK for d in ds)} · deny "
      f"{sum(d.verdict == DENY for d in ds)}")
print(f"tail coverage  P(Y>=lo)={np.mean(ys >= los):.3f}  "
      f"P(Y<=hi)={np.mean(ys <= his):.3f}")
print(f"max lower bound on the test split: {los.max():.1f}  (deny needs > 240)")

dm = dm_1s      # the two-track model governs the report below


band 173.2 · release 41 · ask 46 · deny 0
tail coverage  P(Y>=lo)=0.931  P(Y<=hi)=0.989
max lower bound on the test split: 131.7  (deny needs > 240)


### Risk track: action risk control (CRC)

The coverage band answers *where does Y lie for this patient*. It cannot answer
*is releasing this cohort safe* — that is a different guarantee. `calibrate_risk` fits a crossing
model `p_event(x) = P(Y > 240 | x)` and uses Conformal Risk Control to choose two action thresholds:

```
release_ok  p_event <= lambda_release      loss = 1[Y > 240]
deny_ok     p_event >= lambda_deny         loss = 1[Y <= 240]
```

The guarantee is on the expected risk of the selected cohort, not on a single patient's interval. The
coverage facts (`hi`) and the risk facts (`risk_release`, `risk_deny`) are separate tracks in the same
policy; `assurance` records which one decided each row.

The production policy below **prunes `deny_high_band`**: the feasibility audit proves `lo` cannot clear
240, and `deny_risk` covers that rejection authority. The escalation predicate is exhaustive
(`not (deny or release or hi <= 240)`), so no in-regime row can fall through to `abstain(uncovered)`.


In [8]:
from collections import Counter
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from mjolnir_ml import feasibility

@dsl
class ProgressionPolicyV2(DecisionPolicyBase):
    """Production policy: prune the dead coverage deny, keep an exhaustive ask."""

    facts = ("age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6")

    assert lambda bmi: bmi > -1.0, "bmi_in_range"
    assert lambda bp: bp > -1.0, "bp_in_range"

    # Hard deny first. (The engine also applies severity max, so deny always
    # beats ask/release regardless of source order; this documents precedence.)
    assert rule("deny_risk", when=lambda risk_deny: risk_deny, verdict=DENY)

    # Release on either guarantee. A dual-certified row fires both and the
    # disposition records both basis codes.
    assert rule("release_safe_band", when=lambda hi: hi <= 240.0, verdict=RELEASE)
    assert rule("release_risk", when=lambda risk_release: risk_release,
                verdict=RELEASE)

    # Exhaustive escalation: only when no action cleared. This cannot leave an
    # "uncovered" hole and cannot intercept a deny (deny has severity max).
    assert rule("review_unresolved",
                when=lambda hi, risk_release, risk_deny:
                not (risk_deny or risk_release or hi <= 240.0), verdict=ASK)

crossing = GradientBoostingClassifier(random_state=0).fit(
    Xtr, (ytr > 240).astype(int))
dm_risk = (dm_1s.with_policy(ProgressionPolicyV2)
           .calibrate_risk(cal_rows, ycal, tau=240.0, estimator=crossing,
                           alpha_release=0.10, alpha_deny=0.10))

ds = [dm_risk.dispose(r) for r in test_rows]
print("verdicts  :", dict(Counter(d.verdict for d in ds)))
print("assurance :", dict(Counter(d.assurance for d in ds)))
print("lambdas   : release", round(dm_risk.risk.lambda_release("__all__"), 3),
      "· deny", round(dm_risk.risk.lambda_deny("__all__"), 3))

f = feasibility(dm_risk, test_rows, threshold=240.0)
print("dead rules:", f.dead_rules or "none")
print("max lo    :", round(f.coverage_bounds["max_lo"], 1),
      "· max p_event:", round(f.risk_bounds["max_p_event"], 3))

dm = dm_risk      # the two-track model governs the report below


verdicts  : {'ask': 36, 'release': 50, 'abstain': 2, 'deny': 1}
assurance : {'action_risk_control': 46, 'dual_track': 40, 'declared': 2, 'pointwise_coverage': 1}
lambdas   : release 0.074 · deny 0.983


dead rules: none
max lo    : 131.7 · max p_event: 0.994


## 5. The comprehensive report (problem · decisions · hazards · certificates)

`report()` is the single artifact for a reviewer: the compiled policy, verdict census, abstention causes,
the Ω pair against a pinned oracle (`safe = y <= 240`), completeness against an **external** hazard ledger,
and the certificate census. The ledger is external to the policy: spec limits on BMI and blood pressure,
declared with provenance.

In [9]:
ledger = (HazardLedger()
          .add("extreme_bmi", lambda r: r["bmi"] > 0.10, "clinical spec §2 (BMI)")
          .add("high_bp", lambda r: r["bp"] > 0.14, "clinical spec §3 (blood pressure)"))

safe = [int(v) <= 240 for v in yte]
decidable = [True] * len(yte)

text = report(dm, test_rows, title="Diabetes progression — safe-to-act decision",
              safe=safe, decidable=decidable, ledger=ledger,
              key=b"demo-audit-key", sample=10, threshold=240.0)
print(text)

 Diabetes progression — safe-to-act decision
 mjolnir.ml.report/1 · 2026-10-25T17:37:26Z

CONTRACT (the problem)
----------------------
  policy        ProgressionPolicyV2  (sha256:c8833e8f98ce72ce798b53aad7584cd8a3fe0efa37b151c0a28fc3e65d59bc69)
  model         RandomForestRegressor  (sha256:b2fac7b2d0deb7bf…)
  calibration   alpha=0.10  (sha256:d2f28895fe236088…)
  bounds        one-sided  P(Y>=lo)>=1-0.10 · P(Y<=hi)>=1-0.10  (decision envelope, not a joint interval)
  risk track    event=Y>240  alpha_release=0.10 · alpha_deny=0.10  (sha256:3d14135e0eced36b…)
  assurance     coverage=pointwise_coverage · risk=action_risk_control · both=dual_track
  regime        required  method=mahalanobis  alpha_ood=0.02  tau=1198.82  (sha256:34da619fb38f993e…)
  features      age, sex, bmi, bp, s1, s2, s3, s4, s5, s6

POLICY (every statement, compiled)
----------------------------------
  code               kind     requires                     predicate                                       verdi

## 6. Auditing one decision without the forest

The certificate publishes the **complete fact table** (stated inputs + model
facts — here the conformal interval `lo`/`hi`). An auditor holding the policy
artifact and the key re-derives the verdict with `Policy.dispose_facts` —
**the same function the live `dispose()` ran** — so the live path and the
audit path cannot drift. No RandomForest, no training data, no sklearn
import needed.

Four checks, each fail-closed:

1. **structure** — schema, verdict vocabulary, digest shapes;
2. **signature** — HMAC-SHA256 over the canonical certificate;
3. **binding** — `certificate_id` is *recomputed* from the pinned digests +
   verdict/basis/guarantee, and a supplied `row=` is hashed against
   `claim_digest` (input integrity);
4. **replay** — verdict/basis/missing are re-derived from the published
   facts, and the `guarantee`/`assurance` tags are re-derived from the
   policy — an unsigned certificate cannot relabel them either.

The same path covers **classifiers** unchanged: the policy replays whatever
facts the table carries (`conf`, `p_positive`, `certified`, `set_size`,
`p_event`, …) — see the classifier notebook. One honest limit:
harness-stage abstentions (`out_of_regime*`, `regime_not_fitted`, …) depend
on the training bounds, so they are **signed and bound but attested, not
replayed** — that is the `replayable from facts alone 87/89` line in the
report.

In [10]:
d0 = dm.dispose(test_rows[0])
cert = certify(d0, key=b"demo-audit-key")

# signed verify: key + policy + the row itself (input-integrity check)
print("verdict                :", cert["verdict"], "| basis:", cert["basis"])
print("published facts        :",
      {k: round(v, 3) for k, v in cert["facts"].items()
       if k in ("pred", "lo", "hi")})
print("verified (key+policy+row):",
      verify_certificate(cert, key=b"demo-audit-key",
                         policy=dm.policy, row=test_rows[0]))
print("replayable from facts  :", cert["replayable"])

# 1) signed verdict tamper -> the HMAC rejects it
bad = dict(cert, verdict="release" if cert["verdict"] != "release" else "deny")
try:
    verify_certificate(bad, key=b"demo-audit-key", policy=dm.policy)
except Exception as ex:
    print("tampered verdict (signed):", type(ex).__name__, "—", ex)

# 2) unsigned FACT tamper -> replay re-derives the verdict from the facts
#    (breaking the bmi premise must flip the outcome to an abstain)
bad = dict(certify(d0), facts=dict(cert["facts"], bmi=-5.0))
try:
    verify_certificate(bad, policy=dm.policy)
except Exception as ex:
    print("tampered facts (unsigned):", type(ex).__name__, "—", ex)

# 3) rewritten certificate id -> the binding check recomputes it
from mjolnir_ml.certificate import _binding_id
bad = dict(certify(d0), certificate_id="sha256:" + "0" * 64)
try:
    verify_certificate(bad, policy=dm.policy)
except Exception as ex:
    print("rewritten id (unsigned)  :", type(ex).__name__, "—", ex)

# 4) another row's inputs -> claim_digest mismatch
try:
    verify_certificate(cert, key=b"demo-audit-key", policy=dm.policy,
                       row=test_rows[1])
except Exception as ex:
    print("wrong row (input tamper) :", type(ex).__name__, "—", ex)

verdict                : ask | basis: ['review_unresolved']
published facts        : {'pred': 252.287, 'lo': 111.985, 'hi': 305.722}
verified (key+policy+row): True
replayable from facts  : True
tampered verdict (signed): CertificateError — signature does not match the certificate
tampered facts (unsigned): CertificateError — replay disagrees: certificate says (ask, ['review_unresolved'], None), policy over the published facts says (abstain, ['bmi_in_range'], None)
rewritten id (unsigned)  : CertificateError — certificate_id does not match the pinned digests (binding check failed)
wrong row (input tamper) : CertificateError — input tamper: the supplied row does not hash to the certificate's claim_digest


## Takeaways

- The forest is a **proposal of facts** (`pred`, conformal `lo`/`hi`); the policy is the only disposer.
- A release requires the *upper* bound to clear the band; a negative lower bound is a statistical artifact,
  not a clinical hazard. The three rules cover every interval, so nothing abstains for "uncovered".
- Local-width conformal (normalized / CQR) spends width where the model is noisy instead of paying a
  constant worst-case band; CQR is the tightest here and the same coverage floor + digest discipline hold.
- One-sided tails buy a tighter lower bound, but a `deny` still requires the conditional lower decile to
  clear 240. On this dataset that never happens (max lower bound ≈ 132): the coverage deny rule is DEAD.
- The risk track changes the question: it certifies the *expected risk of the released/denied cohort*,
  so `deny` becomes reachable without pretending a single patient's interval cleared 240. The report
  shows both the dead coverage rule and the reachable CRC rule.
- The report is digest-pinned and the certificate replays without the learner.
- Swap `RandomForestRegressor` for `Ridge` or `HistGradientBoostingRegressor`: policy, κ and certificates
  are unchanged; only `model_digest` moves.